In [1]:
!pip install -q xgboost imbalanced-learn scipy scikit-learn pandas numpy

# Eksperimen lintas dataset untuk prediksi defect
Notebook ini mempertahankan pipeline M2, M4, dan M6, lalu mengevaluasi seluruh dataset dengan repeated stratified cross-validation. Preprocessing dan undersampling hanya dipelajari dari training fold.

In [2]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff
from scipy.stats import wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, average_precision_score, roc_auc_score)
from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from google.colab import drive
drive.mount('/content/drive')

RANDOM_STATE = 42
DATA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/DatasetRM')
DATA_PATHS = {
    # NASA MDP (ARFF)
    'PC1': DATA_ROOT / 'PC1.arff', 'PC2': DATA_ROOT / 'PC2.arff',
    'PC3': DATA_ROOT / 'PC3.arff', 'PC4': DATA_ROOT / 'PC4.arff',
    'PC5': DATA_ROOT / 'PC5.arff', 'KC1': DATA_ROOT / 'KC1.arff',
    'KC3': DATA_ROOT / 'KC3.arff', 'MC1': DATA_ROOT / 'MC1.arff',
    # PROMISE/JIRA-style CSV
    'ivy-1.4': DATA_ROOT / 'ivy-1.4.csv', 'ivy-2.0': DATA_ROOT / 'ivy-2.0.csv',
    'jedit-4.0': DATA_ROOT / 'jedit-4.0.csv', 'jedit-4.2': DATA_ROOT / 'jedit-4.2.csv',
    'Tomcat': DATA_ROOT / 'tomcat.csv', 'synapse-1.0': DATA_ROOT / 'synapse-1.0.csv',
    'xalan-2.4': DATA_ROOT / 'xalan-2.4.csv', 'xerces-1.2': DATA_ROOT / 'xerces-1.2.csv',
}
# NASA MDP memakai Defective Y/N. CSV memakai kolom target yang ditemukan
# dari kandidat di bawah; isikan nama kolom secara eksplisit di sini bila berbeda/ambigu.
TARGET_COLUMNS = {name: ('Defective' if name in {'PC1','PC2','PC3','PC4','PC5','KC1','KC3','MC1'} else None) for name in DATA_PATHS}
TARGET_CANDIDATES = ('defective','bug','bugs','buggy','is_buggy','is_defective','defects','label','class')
HMD_MAX_DEPTH = 3
HMD_MIN_MAJORITY_SIZE = 20
N_SPLITS = 5
N_REPEATS = 5
XGB_PARAMS = {'n_estimators':150,'max_depth':4,'learning_rate':0.05,
    'subsample':0.8,'colsample_bytree':0.8,'eval_metric':'logloss',
    'random_state':RANDOM_STATE,'n_jobs':-1}


Mounted at /content/drive


## Loader dan pemetaan target

In [3]:
def load_dataset(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Dataset tidak ditemukan: {path}')
    if path.suffix.lower() == '.arff':
        raw, _ = arff.loadarff(path)
        frame = pd.DataFrame(raw)
        for col in frame.select_dtypes(include=['object']).columns:
            frame[col] = frame[col].map(lambda v: v.decode('utf-8') if isinstance(v, bytes) else v)
        return frame
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Ekstensi tidak didukung: {path.suffix}')

def resolve_target_column(frame, dataset_name):
    configured = TARGET_COLUMNS.get(dataset_name)
    if configured:
        if configured not in frame.columns:
            raise ValueError(f'{dataset_name}: target {configured!r} tidak ada. Kolom: {list(frame.columns)}')
        return configured
    normalized = {str(c).strip().lower(): c for c in frame.columns}
    matches = [normalized[name] for name in TARGET_CANDIDATES if name in normalized]
    if len(matches) != 1:
        raise ValueError(f'{dataset_name}: target tidak dapat ditentukan secara unik (kandidat cocok: {matches}). Isi TARGET_COLUMNS[{dataset_name!r}] dengan nama kolom yang benar.')
    return matches[0]

def map_defect_target(series, dataset_name):
    values = series.dropna()
    if values.empty:
        raise ValueError(f'{dataset_name}: target seluruhnya kosong')
    # Untuk label teks, terima nilai defect/non-defect yang eksplisit saja.
    normalized = values.astype(str).str.strip().str.lower()
    positive = {'y','yes','true','1','bug','buggy','defect','defective','positive'}
    negative = {'n','no','false','0','clean','non-defect','non_defect','nonbuggy','negative'}
    observed = set(normalized.unique())
    if observed <= positive | negative and observed & positive and observed & negative:
        mapped = series.astype(str).str.strip().str.lower().map(lambda v: 1 if v in positive else (0 if v in negative else np.nan))
        return mapped.astype('Int64').astype(int), sorted(observed)
    numeric = pd.to_numeric(series, errors='coerce')
    unique = set(numeric.dropna().unique())
    if unique == {0, 1}:
        return numeric.astype('Int64').astype(int), sorted(unique)
    # PROMISE defect counts: nol = non-defect, nilai positif = defect.
    if numeric.notna().all() and len(unique) >= 2 and min(unique) >= 0:
        return (numeric > 0).astype(int), sorted(unique)
    raise ValueError(f'{dataset_name}: label target tidak dikenal/ambigu: {sorted(observed)}. Konfirmasi pemetaan target dan sesuaikan map_defect_target.')

def prepare_dataset(name, path):
    frame = load_dataset(path).drop_duplicates().reset_index(drop=True)
    target_col = resolve_target_column(frame, name)
    keep = frame[target_col].notna()
    frame = frame.loc[keep].reset_index(drop=True)
    y, source_labels = map_defect_target(frame[target_col], name)
    X = frame.drop(columns=[target_col])
    counts = y.value_counts().to_dict()
    if set(counts) != {0,1}:
        raise ValueError(f'{name}: setelah pemetaan target tidak ada dua kelas: {counts}')
    return X, y, {'Dataset':name,'Path':str(path),'Rows':len(frame),'Features':X.shape[1],
        'Target column':target_col,'Source labels':str(source_labels),'Non-defect (0)':counts[0],
        'Defect (1)':counts[1],'Status':'ready','Issue':''}


## Model dan evaluasi

In [4]:
def to_dense_array(x):
    return np.asarray(x.toarray() if hasattr(x, 'toarray') else x, dtype=np.float32)

def create_preprocessor(X_fit):
    numeric = X_fit.select_dtypes(include=['number']).columns.tolist()
    categorical = X_fit.select_dtypes(include=['object','category','bool']).columns.tolist()
    transformers = []
    if numeric:
        transformers.append(('numeric', Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]), numeric))
    if categorical:
        transformers.append(('categorical', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), categorical))
    if not transformers:
        raise ValueError('Tidak ada fitur numerik/kategorikal yang dapat diproses.')
    return ColumnTransformer(transformers=transformers, remainder='drop')

def create_xgb_binary(random_state):
    params = XGB_PARAMS.copy(); params.update(objective='binary:logistic', eval_metric='logloss', random_state=random_state)
    return XGBClassifier(**params)

def calculate_gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    sensitivity = tp/(tp+fn) if tp+fn else 0.0
    specificity = tn/(tn+fp) if tn+fp else 0.0
    return float(np.sqrt(sensitivity*specificity))

def evaluate_model(y_true, y_pred, model_name, y_prob=None):
    result = {'Model':model_name,'Precision':precision_score(y_true,y_pred,pos_label=1,zero_division=0),
      'Recall':recall_score(y_true,y_pred,pos_label=1,zero_division=0),'F1-score':f1_score(y_true,y_pred,pos_label=1,zero_division=0),
      'G-Mean':calculate_gmean(y_true,y_pred),'Balanced Accuracy':balanced_accuracy_score(y_true,y_pred),
      'MCC':matthews_corrcoef(y_true,y_pred)}
    if y_prob is not None:
        result['AUC-ROC']=roc_auc_score(y_true,y_prob)
        result['AUC-PR']=average_precision_score(y_true,y_prob)
    return result


## HMD + local XGBoost

In [5]:
class HMDXGBoost:
    """
    Hierarchical Majority Decomposition + XGBoost.

    Target global:
        0 = non-defect / majority
        1 = defect / minority

    Internal node:
        0 = defect
        1 = majority child 1
        2 = majority child 2

    Leaf:
        0 = non-defect
        1 = defect
    """

    def __init__(
        self,
        max_depth=3,
        min_majority_size=20,
        random_state=42,
        xgb_params=None
    ):
        self.max_depth = max_depth
        self.min_majority_size = min_majority_size
        self.random_state = random_state

        self.xgb_params = xgb_params or XGB_PARAMS.copy()

        self.root = None
        self.X_train = None
        self.y_train = None
        self.minority_indices = None

    def _make_binary_model(self, random_state):
        params = self.xgb_params.copy()

        params.update({
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": random_state
        })

        params.pop("num_class", None)

        return XGBClassifier(**params)

    def _make_multiclass_model(self, random_state):
        params = self.xgb_params.copy()

        params.update({
            "objective": "multi:softprob",
            "num_class": 3,
            "eval_metric": "mlogloss",
            "random_state": random_state
        })

        return XGBClassifier(**params)

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y, dtype=int)

        self.minority_indices = np.where(
            self.y_train == 1
        )[0]

        majority_indices = np.where(
            self.y_train == 0
        )[0]

        if len(self.minority_indices) == 0:
            raise ValueError(
                "Tidak ditemukan sampel defect pada training data."
            )

        if len(majority_indices) < 2:
            raise ValueError(
                "Jumlah sampel mayoritas tidak cukup untuk K-Means."
            )

        self.root = self._build_node(
            majority_indices=majority_indices,
            depth=0
        )

        return self

    def _build_node(self, majority_indices, depth):
        node = {
            "depth": depth,
            "majority_indices": np.asarray(majority_indices),
            "is_leaf": False,
            "model": None,
            "left": None,
            "right": None,
            "kmeans": None
        }

        # Kondisi berhenti: node menjadi leaf
        stop_decomposition = (
            depth >= self.max_depth
            or len(majority_indices) < self.min_majority_size
            or len(majority_indices) < 2
        )

        if stop_decomposition:
            node["is_leaf"] = True

            # Data leaf:
            # minority defect + majority yang berada di leaf.
            leaf_indices = np.concatenate([
                self.minority_indices,
                majority_indices
            ])

            X_leaf = self.X_train[leaf_indices]
            y_leaf = self.y_train[leaf_indices]

            # Classifier leaf memprediksi:
            # 0 = non-defect
            # 1 = defect
            leaf_model = self._make_binary_model(
                self.random_state + depth
            )

            leaf_model.fit(X_leaf, y_leaf)

            node["model"] = leaf_model

            return node

        # K-Means hanya diterapkan pada kelas mayoritas
        X_majority = self.X_train[majority_indices]

        kmeans = KMeans(
            n_clusters=2,
            random_state=self.random_state + depth,
            n_init=10
        )

        cluster_labels = kmeans.fit_predict(X_majority)

        cluster_0_indices = majority_indices[
            cluster_labels == 0
        ]

        cluster_1_indices = majority_indices[
            cluster_labels == 1
        ]

        # Jika salah satu cluster kosong, gunakan leaf
        if (
            len(cluster_0_indices) == 0
            or len(cluster_1_indices) == 0
        ):
            node["is_leaf"] = True

            leaf_indices = np.concatenate([
                self.minority_indices,
                majority_indices
            ])

            X_leaf = self.X_train[leaf_indices]
            y_leaf = self.y_train[leaf_indices]

            leaf_model = self._make_binary_model(
                self.random_state + depth
            )

            leaf_model.fit(X_leaf, y_leaf)

            node["model"] = leaf_model

            return node

        node["kmeans"] = kmeans

        # Dataset lokal internal node:
        # seluruh minority + majority child 1 + majority child 2.
        local_indices = np.concatenate([
            self.minority_indices,
            cluster_0_indices,
            cluster_1_indices
        ])

        X_local = self.X_train[local_indices]

        y_local = np.concatenate([
            np.zeros(
                len(self.minority_indices),
                dtype=int
            ),
            np.ones(
                len(cluster_0_indices),
                dtype=int
            ),
            np.full(
                len(cluster_1_indices),
                2,
                dtype=int
            )
        ])

        local_model = self._make_multiclass_model(
            self.random_state + depth
        )

        local_model.fit(
            X_local,
            y_local
        )

        node["model"] = local_model

        # Recursive decomposition pada kelas mayoritas
        node["left"] = self._build_node(
            majority_indices=cluster_0_indices,
            depth=depth + 1
        )

        node["right"] = self._build_node(
            majority_indices=cluster_1_indices,
            depth=depth + 1
        )

        return node

    def _predict_one(self, x, node):
        # Leaf: klasifikasi defect vs non-defect
        if node["is_leaf"]:
            pred = int(
                node["model"].predict(
                    x.reshape(1, -1)
                )[0]
            )

            return pred

        # Internal node: prediksi kelas lokal
        local_pred = int(
            node["model"].predict(
                x.reshape(1, -1)
            )[0]
        )

        # Local label 0 = defect
        if local_pred == 0:
            return 1

        # Local label 1 = majority child 1
        if local_pred == 1:
            return self._predict_one(
                x,
                node["left"]
            )

        # Local label 2 = majority child 2
        if local_pred == 2:
            return self._predict_one(
                x,
                node["right"]
            )

        # Fallback defensif
        return 0

    def predict(self, X):
        X = np.asarray(X)

        predictions = []

        for x in X:
            predictions.append(
                self._predict_one(x, self.root)
            )

        return np.asarray(
            predictions,
            dtype=int
        )

    def predict_proba(self, X):
        """
        Menghasilkan probabilitas defect dari model leaf
        yang dicapai oleh masing-masing sampel.

        Ini adalah probabilitas lokal pada leaf, bukan
        probabilitas global yang dikalibrasi sepanjang jalur.
        """
        X = np.asarray(X)

        probabilities = []

        for x in X:
            node = self.root

            while not node["is_leaf"]:
                local_pred = int(
                    node["model"].predict(
                        x.reshape(1, -1)
                    )[0]
                )

                if local_pred == 0:
                    # Internal node memprediksi defect.
                    # Nilai ini merupakan confidence lokal
                    # dari kelas defect pada node.
                    local_prob = node["model"].predict_proba(
                        x.reshape(1, -1)
                    )[0][0]

                    probabilities.append(
                        float(local_prob)
                    )

                    break

                elif local_pred == 1:
                    node = node["left"]

                else:
                    node = node["right"]

            else:
                # Sampel mencapai leaf
                prob = node["model"].predict_proba(
                    x.reshape(1, -1)
                )[0][1]

                probabilities.append(
                    float(prob)
                )

        return np.asarray(probabilities)

## Jalankan semua dataset: preprocessing, RUS, dan HMD di dalam setiap fold

In [6]:
cv_results = []
audit_rows = []
model_names = ['M2 - XGBoost','M4 - XGBoost + Random Undersampling','M6 - HMD + XGBoost lokal']

for dataset_number, (dataset_name, dataset_path) in enumerate(DATA_PATHS.items(), start=1):
    print(f'\n[{dataset_number}/{len(DATA_PATHS)}] {dataset_name}: {dataset_path}')
    try:
        X, y, audit = prepare_dataset(dataset_name, dataset_path)
        if y.value_counts().min() < N_SPLITS:
            raise ValueError(f'Sampel kelas minoritas ({y.value_counts().min()}) kurang dari N_SPLITS={N_SPLITS}')
        audit_rows.append(audit)
        dataset_cv = RepeatedStratifiedKFold(n_splits=N_SPLITS,n_repeats=N_REPEATS,random_state=RANDOM_STATE)
        for fold_number, (train_idx, valid_idx) in enumerate(dataset_cv.split(X,y), start=1):
            X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx].to_numpy(dtype=int), y.iloc[valid_idx].to_numpy(dtype=int)
            seed = RANDOM_STATE + dataset_number*1000 + fold_number
            # Imputer, scaler, dan encoder di-fit hanya pada training fold.
            preprocessor = create_preprocessor(X_train)
            X_train_p = to_dense_array(preprocessor.fit_transform(X_train))
            X_valid_p = to_dense_array(preprocessor.transform(X_valid))

            m2 = create_xgb_binary(seed).fit(X_train_p,y_train)
            pred_m2 = m2.predict(X_valid_p); prob_m2 = m2.predict_proba(X_valid_p)[:,1]

            # RUS melihat training fold saja; validation fold tetap pada distribusi asli.
            X_rus,y_rus = RandomUnderSampler(random_state=seed).fit_resample(X_train_p,y_train)
            m4 = create_xgb_binary(seed).fit(X_rus,y_rus)
            pred_m4 = m4.predict(X_valid_p); prob_m4 = m4.predict_proba(X_valid_p)[:,1]

            # Objek HMD baru diinisialisasi dan di-fit ulang untuk setiap fold.
            m6 = HMDXGBoost(max_depth=HMD_MAX_DEPTH,min_majority_size=HMD_MIN_MAJORITY_SIZE,random_state=seed,xgb_params=XGB_PARAMS)
            m6.fit(X_train_p,y_train)
            pred_m6 = m6.predict(X_valid_p)

            fold_outputs = [(model_names[0],pred_m2,prob_m2),(model_names[1],pred_m4,prob_m4),(model_names[2],pred_m6,None)]
            for model_name,pred,prob in fold_outputs:
                row = evaluate_model(y_valid,pred,model_name,y_prob=prob)
                row.update({'Dataset':dataset_name,'Fold':fold_number})
                cv_results.append(row)
        print(f'  selesai: {N_SPLITS*N_REPEATS} folds')
    except Exception as exc:
        # Catat masalah per dataset agar dataset lain tetap dapat diproses.
        audit_rows.append({'Dataset':dataset_name,'Path':str(dataset_path),'Rows':None,'Features':None,
            'Target column':TARGET_COLUMNS.get(dataset_name),'Source labels':None,'Non-defect (0)':None,'Defect (1)':None,
            'Status':'error','Issue':str(exc)})
        print(f'  dilewati: {exc}')

cv_results_df = pd.DataFrame(cv_results)
audit_df = pd.DataFrame(audit_rows)
if cv_results_df.empty:
    raise RuntimeError('Tidak ada dataset yang berhasil dievaluasi. Periksa audit_dataset.csv dan pemetaan target.')
display(audit_df)
display(cv_results_df.head())



[1/16] PC1: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC1.arff
  selesai: 25 folds

[2/16] PC2: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC2.arff
  selesai: 25 folds

[3/16] PC3: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC3.arff
  selesai: 25 folds

[4/16] PC4: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC4.arff
  selesai: 25 folds

[5/16] PC5: /content/drive/MyDrive/Colab Notebooks/DatasetRM/PC5.arff
  selesai: 25 folds

[6/16] KC1: /content/drive/MyDrive/Colab Notebooks/DatasetRM/KC1.arff
  selesai: 25 folds

[7/16] KC3: /content/drive/MyDrive/Colab Notebooks/DatasetRM/KC3.arff
  selesai: 25 folds

[8/16] MC1: /content/drive/MyDrive/Colab Notebooks/DatasetRM/MC1.arff
  selesai: 25 folds

[9/16] ivy-1.4: /content/drive/MyDrive/Colab Notebooks/DatasetRM/ivy-1.4.csv
  selesai: 25 folds

[10/16] ivy-2.0: /content/drive/MyDrive/Colab Notebooks/DatasetRM/ivy-2.0.csv
  selesai: 25 folds

[11/16] jedit-4.0: /content/drive/MyDrive/Colab Notebooks/DatasetRM/jedit-

,Dataset,Path,Rows,Features,Target column,Source labels,Non-defect (0),Defect (1),Status,Issue
0,PC1,/content/drive/MyDrive/Colab Notebooks/Dataset...,711,37,Defective,"['n', 'y']",650,61,ready,
1,PC2,/content/drive/MyDrive/Colab Notebooks/Dataset...,745,36,Defective,"['n', 'y']",729,16,ready,
2,PC3,/content/drive/MyDrive/Colab Notebooks/Dataset...,1079,37,Defective,"['n', 'y']",945,134,ready,
3,PC4,/content/drive/MyDrive/Colab Notebooks/Dataset...,1288,37,Defective,"['n', 'y']",1111,177,ready,
4,PC5,/content/drive/MyDrive/Colab Notebooks/Dataset...,1723,38,Defective,"['n', 'y']",1252,471,ready,
5,KC1,/content/drive/MyDrive/Colab Notebooks/Dataset...,1203,21,Defective,"['n', 'y']",889,314,ready,
6,KC3,/content/drive/MyDrive/Colab Notebooks/Dataset...,194,39,Defective,"['n', 'y']",158,36,ready,
7,MC1,/content/drive/MyDrive/Colab Notebooks/Dataset...,1998,38,Defective,"['n', 'y']",1952,46,ready,
8,ivy-1.4,/content/drive/MyDrive/Colab Notebooks/Dataset...,241,23,bug,"[np.int64(0), np.int64(1), np.int64(3)]",225,16,ready,
9,ivy-2.0,/content/drive/MyDrive/Colab Notebooks/Dataset...,352,23,bug,"[np.int64(0), np.int64(1), np.int64(2), np.int...",312,40,ready,


,Model,Precision,Recall,F1-score,G-Mean,Balanced Accuracy,MCC,AUC-ROC,AUC-PR,Dataset,Fold
0,M2 - XGBoost,0.500000,0.153846,0.235294,0.389203,0.569231,0.241399,0.861834,0.439315,PC1,1
1,M4 - XGBoost + Random Undersampling,0.204545,0.692308,0.315789,0.711279,0.711538,0.263523,0.857396,0.557183,PC1,1
2,M6 - HMD + XGBoost lokal,0.363636,0.307692,0.333333,0.539559,0.626923,0.273861,NaN,NaN,PC1,1
3,M2 - XGBoost,0.333333,0.166667,0.222222,0.401918,0.567949,0.187901,0.827885,0.326159,PC1,2
4,M4 - XGBoost + Random Undersampling,0.177778,0.666667,0.280702,0.690596,0.691026,0.228398,0.795833,0.202409,PC1,2


## Ringkasan per dataset dan model

In [7]:
metric_columns = ['Precision','Recall','F1-score','G-Mean','Balanced Accuracy','MCC','AUC-ROC','AUC-PR']
summary_by_dataset = cv_results_df.groupby(['Dataset','Model'])[metric_columns].agg(['mean','std'])
summary_overall = cv_results_df.groupby('Model')[metric_columns].agg(['mean','std'])
display(summary_by_dataset)
display(summary_overall)


Precision              Recall  \
                                                     mean       std      mean   
Dataset     Model                                                               
KC1         M2 - XGBoost                         0.606444  0.077609  0.256119   
            M4 - XGBoost + Random Undersampling  0.375758  0.026999  0.582284   
            M6 - HMD + XGBoost lokal             0.482927  0.057563  0.406523   
KC3         M2 - XGBoost                         0.406762  0.323284  0.182143   
            M4 - XGBoost + Random Undersampling  0.290570  0.055934  0.623571   
            M6 - HMD + XGBoost lokal             0.341662  0.087769  0.478571   
MC1         M2 - XGBoost                         0.120000  0.258064  0.033333   
            M4 - XGBoost + Random Undersampling  0.062733  0.009955  0.796000   
            M6 - HMD + XGBoost lokal             0.204238  0.237881  0.089778   
PC1         M2 - XGBoost                         0.439143  0.221580  0.173846   
            M4 - XGBoost + Random Undersampling  0.212574  0.036417  0.781026   
            M6 - HMD + XGBoost lokal             0.327875  0.132581  0.361795   
PC2         M2 - XGBoost                         0.000000  0.000000  0.000000   
            M4 - XGBoost + Random Undersampling  0.055873  0.015943  0.716667   
            M6 - HMD + XGBoost lokal             0.040000  0.200000  0.013333   
PC3         M2 - XGBoost                         0.435588  0.153516  0.165641   
            M4 - XGBoost + Random Undersampling  0.275309  0.020715  0.783761   
            M6 - HMD + XGBoost lokal             0.395945  0.087052  0.320912   
PC4         M2 - XGBoost                         0.679576  0.076245  0.482635   
            M4 - XGBoost + Random Undersampling  0.423451  0.037006  0.898540   
            M6 - HMD + XGBoost lokal             0.552589  0.058676  0.616825   
PC5         M2 - XGBoost                         0.619186  0.060123  0.382598   
            M4 - XGBoost + Random Undersampling  0.460898  0.027416  0.720945   
            M6 - HMD + XGBoost lokal             0.555569  0.050885  0.496770   
Tomcat      M2 - XGBoost                         0.471837  0.182622  0.184500   
            M4 - XGBoost + Random Undersampling  0.233990  0.030588  0.743667   
            M6 - HMD + XGBoost lokal             0.300184  0.104326  0.296167   
ivy-1.4     M2 - XGBoost                         0.000000  0.000000  0.000000   
            M4 - XGBoost + Random Undersampling  0.103099  0.052086  0.620000   
            M6 - HMD + XGBoost lokal             0.049778  0.101233  0.086667   
ivy-2.0     M2 - XGBoost                         0.420048  0.223014  0.210000   
            M4 - XGBoost + Random Undersampling  0.240822  0.047379  0.730000   
            M6 - HMD + XGBoost lokal             0.356941  0.105853  0.370000   
jedit-4.0   M2 - XGBoost                         0.705270  0.135934  0.501333   
            M4 - XGBoost + Random Undersampling  0.485544  0.079884  0.712000   
            M6 - HMD + XGBoost lokal             0.531057  0.089304  0.610667   
jedit-4.2   M2 - XGBoost                         0.567016  0.235932  0.270222   
            M4 - XGBoost + Random Undersampling  0.318381  0.052602  0.761778   
            M6 - HMD + XGBoost lokal             0.435294  0.124316  0.491556   
synapse-1.0 M2 - XGBoost                         0.142667  0.260078  0.103333   
            M4 - XGBoost + Random Undersampling  0.240644  0.094343  0.723333   
            M6 - HMD + XGBoost lokal             0.160654  0.147191  0.223333   
xalan-2.4   M2 - XGBoost                         0.539621  0.178469  0.245455   
            M4 - XGBoost + Random Undersampling  0.296323  0.032651  0.729091   
            M6 - HMD + XGBoost lokal             0.377261  0.071230  0.509091   
xerces-1.2  M2 - XGBoost                         0.532105  0.176654  0.251048   
            M4 - XGBoost + Random Undersampling  0.304004  0.054108  0.684381  

Precision              Recall            \
                                         mean       std      mean       std   
Model                                                                         
M2 - XGBoost                         0.417829  0.287595  0.215138  0.173483   
M4 - XGBoost + Random Undersampling  0.273748  0.132360  0.725440  0.176437   
M6 - HMD + XGBoost lokal             0.343710  0.196171  0.361273  0.213787   

                                     F1-score              G-Mean            \
                                         mean       std      mean       std   
Model                                                                         
M2 - XGBoost                         0.273624  0.201177  0.380286  0.244144   
M4 - XGBoost + Random Undersampling  0.377451  0.153388  0.708205  0.110476   
M6 - HMD + XGBoost lokal             0.342625  0.191384  0.505364  0.244522   

                                    Balanced Accuracy                 MCC  \
                                                 mean       std      mean   
Model                                                                       
M2 - XGBoost                                 0.588506  0.079757  0.231977   
M4 - XGBoost + Random Undersampling          0.717847  0.092453  0.294676   
M6 - HMD + XGBoost lokal                     0.627414  0.093185  0.250200   

                                                AUC-ROC              AUC-PR  \
                                          std      mean       std      mean   
Model                                                                         
M2 - XGBoost                         0.191207  0.791973  0.101741  0.425816   
M4 - XGBoost + Random Undersampling  0.136735  0.778005  0.098863  0.377522   
M6 - HMD + XGBoost lokal             0.177246       NaN       NaN       NaN   

                                               
                                          std  
Model                                          
M2 - XGBoost                         0.172421  
M4 - XGBoost + Random Undersampling  0.169977  
M6 - HMD + XGBoost lokal                  NaN

## Uji Wilcoxon berpasangan per dataset

In [8]:
def paired_wilcoxon_by_dataset(cv_df, model_a, model_b, metric='F1-score'):
    rows=[]
    for dataset, group in cv_df.groupby('Dataset'):
        a=group[group.Model==model_a].sort_values('Fold')[metric].to_numpy()
        b=group[group.Model==model_b].sort_values('Fold')[metric].to_numpy()
        if len(a)!=len(b) or len(a)<2 or np.allclose(a,b):
            rows.append({'Dataset':dataset,'Model A':model_a,'Model B':model_b,'Metric':metric,'p-value':np.nan,'Note':'tidak cukup pasangan atau seluruh skor sama'})
            continue
        stat,p=wilcoxon(a,b,alternative='two-sided')
        rows.append({'Dataset':dataset,'Model A':model_a,'Model B':model_b,'Metric':metric,'Statistic':stat,'p-value':p,'Note':''})
    return pd.DataFrame(rows)

wilcoxon_m6_m2 = paired_wilcoxon_by_dataset(cv_results_df,model_names[0],model_names[2])
wilcoxon_m6_m4 = paired_wilcoxon_by_dataset(cv_results_df,model_names[1],model_names[2])
display(wilcoxon_m6_m2); display(wilcoxon_m6_m4)


,Dataset,Model A,Model B,Metric,Statistic,p-value,Note
0,KC1,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,5.0,5.960464e-07,
1,KC3,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,36.0,6.643633e-04,
2,MC1,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,7.0,1.193172e-02,
3,PC1,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,24.0,3.182172e-04,
4,PC2,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,0.0,3.173105e-01,
5,PC3,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,8.0,1.490116e-06,
6,PC4,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,101.0,1.013974e-01,
7,PC5,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,3.0,2.980232e-07,
8,Tomcat,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,101.0,1.013974e-01,
9,ivy-1.4,M2 - XGBoost,M6 - HMD + XGBoost lokal,F1-score,0.0,2.728117e-02,


,Dataset,Model A,Model B,Metric,Statistic,p-value,Note
0,KC1,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,113.0,1.908143e-01,
1,KC3,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,157.0,8.823469e-01,
2,MC1,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,154.0,8.325094e-01,
3,PC1,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,144.0,6.186247e-01,
4,PC2,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,25.0,2.140090e-04,
5,PC3,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,48.0,1.296341e-03,
6,PC4,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,125.0,3.253919e-01,
7,PC5,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,5.0,5.960464e-07,
8,Tomcat,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,69.0,1.187255e-02,
9,ivy-1.4,M4 - XGBoost + Random Undersampling,M6 - HMD + XGBoost lokal,F1-score,32.0,1.261733e-03,


## Audit dan ekspor hasil

In [ ]:
OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/DatasetRM/hasil_eksperimen')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
audit_df.to_csv(OUTPUT_DIR/'audit_dataset.csv',index=False)
cv_results_df.to_csv(OUTPUT_DIR/'hasil_repeated_stratified_cv.csv',index=False)
summary_by_dataset.to_csv(OUTPUT_DIR/'ringkasan_cv_per_dataset.csv')
summary_overall.to_csv(OUTPUT_DIR/'ringkasan_repeated_stratified_cv.csv')
wilcoxon_m6_m2.to_csv(OUTPUT_DIR/'wilcoxon_m6_vs_m2_per_dataset.csv',index=False)
wilcoxon_m6_m4.to_csv(OUTPUT_DIR/'wilcoxon_m6_vs_m4_per_dataset.csv',index=False)
print(f'Hasil tersimpan di {OUTPUT_DIR}')
print('Status audit:',audit_df['Status'].value_counts(dropna=False).to_dict())
